# Parse PubMed Abstract-Format Export

Generates JSONL with metadata + `text_to_embed` (title + abstract).

In [13]:
from __future__ import annotations

import json
import re
from pathlib import Path

INPUT_PATH = Path("/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/data/abstract-semaglutid-set.txt")
OUTPUT_PATH = Path("/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/preprocessing_output/semaglutide_pubmed.jsonl")

NOISE_BLOCK_HEADERS = {
    "Author information:",
    "Comment in",
    "Collaborators:",
    "Conflict of interest statement",
    "Erratum in",
    "Retraction in",
    "Retraction of",
    "Updated by",
    "Update in",
}

END_HEADERS = (
    "DOI:",
    "PMID:",
    "PMCID:",
    "Copyright",
    "Publication types",
    "MeSH Terms",
    "Substances",
)

ABSTRACT_LABEL_RE = re.compile(r"^[A-Z][A-Z /\-]{2,}:\s*")
DOI_RE = re.compile(r"\b10\.\d{4,9}/\S+", re.IGNORECASE)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")


In [14]:
def split_records(text: str) -> list[str]:
    starts = [m.start() for m in re.finditer(r"(?m)^\d+\.\s", text)]
    if not starts:
        return []
    starts.append(len(text))
    records = []
    for i in range(len(starts) - 1):
        records.append(text[starts[i] : starts[i + 1]].strip())
    return records


def split_sections_with_ranges(lines: list[str]) -> list[tuple[int, int]]:
    ranges = []
    start = None
    for i, ln in enumerate(lines):
        if ln.strip() == "":
            if start is not None:
                ranges.append((start, i - 1))
                start = None
            continue
        if start is None:
            start = i
    if start is not None:
        ranges.append((start, len(lines) - 1))
    return ranges


def parse_record(block: str) -> dict:
    lines = [ln.rstrip() for ln in block.splitlines()]

    if lines and re.match(r"^\d+\.\s", lines[0]):
        lines[0] = re.sub(r"^\d+\.\s", "", lines[0], count=1)

    ranges = split_sections_with_ranges(lines)
    def block_from_range(idx: int):
        if len(ranges) > idx:
            s, e = ranges[idx]
            return lines[s:e+1], e + 1
        return [], 0

    citation_block, citation_end = block_from_range(0)
    title_block, title_end = block_from_range(1)
    authors_block, authors_end = block_from_range(2)

    citation = " ".join([ln.strip() for ln in citation_block]).strip() or None
    title = " ".join([ln.strip() for ln in title_block]).strip() or None

    year = None
    journal = None
    if citation:
        ym = YEAR_RE.search(citation)
        if ym:
            year = ym.group(0)
            journal = citation[: ym.start()].strip().rstrip(".")
        else:
            journal = citation.strip().rstrip(".")

    pmid = None
    doi = None
    for ln in lines:
        if ln.startswith("PMID:"):
            pmid = re.sub(r"\D", "", ln)
        if doi is None:
            dm = DOI_RE.search(ln)
            if dm:
                doi = dm.group(0).rstrip(".;")

    abstract_lines = []
    in_noise_block = False
    in_abstract = False

    start_idx = authors_end or title_end or citation_end or 0

    for i in range(start_idx, len(lines)):
        raw = lines[i]
        line = raw.strip()

        if not line:
            if in_noise_block:
                in_noise_block = False
            elif in_abstract:
                abstract_lines.append("")
            continue

        if any(line.startswith(h) for h in NOISE_BLOCK_HEADERS):
            in_noise_block = True
            continue

        if line.startswith(END_HEADERS):
            if in_abstract:
                break
            continue

        if in_noise_block:
            continue

        if not in_abstract:
            if ABSTRACT_LABEL_RE.match(line):
                in_abstract = True
                abstract_lines.append(line)
                continue
            in_abstract = True
            abstract_lines.append(line)
            continue

        abstract_lines.append(line)

    abstract = "\n".join(abstract_lines).strip()

    text_to_embed = title or ""
    if abstract:
        text_to_embed = f"{text_to_embed}\n\n{abstract}" if text_to_embed else abstract

    return {
        "pmid": pmid,
        "doi": doi,
        "year": year,
        "journal": journal,
        "title": title,
        "abstract": abstract,
        "text_to_embed": text_to_embed,
    }


In [15]:
text = INPUT_PATH.read_text(errors="ignore")
records = split_records(text)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

parsed = [parse_record(r) for r in records]
filtered = [r for r in parsed if r.get("abstract")]
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for rec in filtered:
        f.write(json.dumps(rec, ensure_ascii=True) + "\n")

missing_pmid = sum(1 for r in parsed if not r.get("pmid"))
missing_doi = sum(1 for r in parsed if not r.get("doi"))
missing_abs = sum(1 for r in parsed if not r.get("abstract"))

print(f"Records: {len(parsed)}")
print(f"Missing PMID: {missing_pmid}")
print(f"Missing DOI: {missing_doi}")
print(f"Missing abstract: {missing_abs}")
print(f"Written (non-empty abstract): {len(filtered)}")
print(f"Output: {OUTPUT_PATH}")


Records: 1663
Missing PMID: 86
Missing DOI: 23
Missing abstract: 91
Written (non-empty abstract): 1572
Output: /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/preprocessing_output/semaglutide_pubmed.jsonl
